# Lab 4.10 &mdash; Fine-tune a Real Sentiment Model (before vs after)

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 40 min &nbsp;|&nbsp; **Day 2 &middot; Module 4 &mdash; Pre-trained Models & Fine-tuning**

### What you'll do
- Load a real model with a fresh (untrained) classification head
- Fine-tune it with a real torch training loop on labelled data
- Measure held-out accuracy BEFORE vs AFTER and predict new text

> **How this lab works (near-real):** these labs use **real Hugging Face Transformers** locally on CPU &mdash; a real pretrained sentiment model, a real tokenizer, and (the headline) a **real fine-tune** you run yourself. Read the **Concept**, fill the real `___` blanks in **Build it** (real model / tokenizer / training-loop calls), **Run it for real** to see the **actual model output** (including the real **before &rarr; after** fine-tune numbers), note **What to notice**, then finish with an open **Your turn**. There is **no auto-grader** &mdash; the goal is real results you can read. The genuine maths (softmax, precision/recall) you still compute **by hand** &mdash; real mechanics, not a stub.

> **Models:** small, CPU-friendly models from the HF hub &mdash; `distilbert-base-uncased-finetuned-sst-2-english` (out-of-the-box sentiment + logits), `distilbert-base-uncased` (tokenizer), `prajjwal1/bert-tiny` (frozen features **and** the model you fine-tune). First use downloads the weights (needs network), then they are cached. An optional hosted comparison uses `ChatGroq` (`GROQ_API_KEY` in `.env`).

**Reference:** [Module 4 slides &mdash; Fine-tuning end-to-end](../../presentation/day2-module4-pretrained-models-and-fine-tuning.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 4 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, pathlib
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True), override=True)   # GROQ_API_KEY etc. (optional hosted compare)

WORK = os.path.join(os.environ.get("TEMP") or os.environ.get("TMP") or "/tmp", "biaa-lab-04-10")
os.makedirs(WORK, exist_ok=True)
print("WORK:", WORK)
print("Real Hugging Face models load from the hub on first use (one-time download, then cached).")

## Concept
This is the client's **fine-tune for sentiment**, for real. We load **prajjwal1/bert-tiny** with a
fresh 2-class head (randomly initialised &mdash; so it starts at chance), then run a real **training
loop**: forward pass &rarr; loss &rarr; `backward()` &rarr; optimizer `step()`, updating **the whole
model**. We measure held-out accuracy **before** (random head) vs **after** training. On CPU this
finishes in a few seconds because bert-tiny is small and the words recur.

## Build it
Fill in the model's label count and the two lines that actually train it.

In [ ]:
import torch, numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification

def load_model():
    name = "prajjwal1/bert-tiny"
    tok = AutoTokenizer.from_pretrained(name)
    model = AutoModelForSequenceClassification.from_pretrained(name, num_labels=___)   # TODO: 2 (pos/neg)
    return tok, model

def evaluate(model, tok, texts, y):
    model.eval()
    with torch.no_grad():
        enc = tok(texts, padding=True, truncation=True, return_tensors="pt")
        pred = model(**enc).logits.argmax(___).numpy()   # TODO: -1  (argmax over classes)
    return float((pred == np.array(y)).mean())

def fine_tune(model, tok, texts, y, steps=40, lr=5e-3):
    enc = tok(texts, padding=True, truncation=True, return_tensors="pt")
    yt = torch.tensor(y)
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    model.train()
    for step in range(steps):
        opt.zero_grad()
        out = model(**enc, labels=yt)   # HF models compute the loss when given labels
        ___   # TODO: out.loss.backward()   -- backprop the gradients
        ___   # TODO: opt.step()            -- update every weight
    return float(out.loss)

## Run it for real
Split, measure BEFORE, fine-tune for real, measure AFTER, then predict new text.

In [ ]:
try:
    # A tiny labelled sentiment dataset (1 = positive, 0 = negative)
    SENT = [
        ("i love this it is great", 1), ("a great and wonderful film", 1),
        ("truly wonderful i love it", 1), ("excellent and brilliant work", 1),
        ("the best most brilliant story", 1), ("i love how great it is", 1),
        ("wonderful excellent and great fun", 1), ("a brilliant and great success", 1),
        ("great fun i really love it", 1), ("the best film wonderful and brilliant", 1),
        ("excellent great and lovely work", 1), ("i love this brilliant great film", 1),
        ("wonderful great and the best", 1), ("so good i love it great", 1),
        ("i hate this it is terrible", 0), ("a terrible and awful film", 0),
        ("truly awful i hate it", 0), ("boring and terrible work", 0),
        ("the worst most boring story", 0), ("i hate how bad it is", 0),
        ("awful boring and dull mess", 0), ("a terrible and bad failure", 0),
        ("boring mess i really hate it", 0), ("the worst film awful and boring", 0),
        ("terrible bad and dull work", 0), ("i hate this awful boring film", 0),
        ("awful terrible and the worst", 0), ("so bad i hate it terrible", 0),
    ]
    texts  = [t for t, y in SENT]
    labels = [y for t, y in SENT]
    from sklearn.model_selection import train_test_split
    Xtr, Xval, ytr, yval = train_test_split(texts, labels, test_size=0.3, random_state=0, stratify=labels)

    torch.manual_seed(0)
    tok, model = load_model()
    before = evaluate(model, tok, Xval, yval)
    final_loss = fine_tune(model, tok, Xtr, ytr)
    after = evaluate(model, tok, Xval, yval)
    print(f"BEFORE fine-tuning (random head):  val acc = {before:.3f}")
    print(f"AFTER  fine-tuning (final loss {final_loss:.3f}): val acc = {after:.3f}")
    print(f"improvement: {after - before:+.3f}")

    for s in ["a wonderful brilliant film", "a boring awful mess", "i really loved every minute"]:
        enc = tok([s], return_tensors="pt")
        with torch.no_grad(): p = int(model(**enc).logits.argmax(-1))
        print(f"  pred={p} ({'pos' if p==1 else 'neg'})  <-  {s}")
except Exception as e:
    print("(If a ___ is still unfilled, fill it above. On first run the model downloads")
    print(" from the Hugging Face hub -- that needs network; re-run once it finishes.)")
    print("  reason:", type(e).__name__, "--", e)

## What to notice
- **BEFORE** the head is random, so val accuracy sits around chance (~0.5). **AFTER** a few seconds of real training it jumps &mdash; a genuine before &rarr; after you produced.
- Unlike Lab 4.7 (frozen backbone), here `backward()` + `step()` update **every weight in the model** &mdash; that is full fine-tuning.
- The loss **falls** across steps; the held-out jump proves the model **generalised**, not just memorised the training rows.

## Your turn (open task &mdash; no grader)
Change `steps` and `lr` in `fine_tune` &mdash; can you make it converge faster, or break it
(too-high `lr` diverges)? Add your own labelled sentences and re-run. A "good" answer: you can state
what `loss.backward()` and `opt.step()` each do, and show one setting that trains and one that doesn't.

---
### Key takeaway
You fine-tuned a **real** model end-to-end: random head &rarr; a trained model that beats its old self on unseen data. That is the client's headline, done for real.

[&#8592; All Module 4 labs](./index.html) &nbsp;&middot;&nbsp; [Module 4 slides](../../presentation/day2-module4-pretrained-models-and-fine-tuning.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>